In [1]:
# Import dependencies
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, pipeline
from transformers import DataCollatorForTokenClassification
from tokenizers import Tokenizer, pre_tokenizers
from tokenizers.pre_tokenizers import Whitespace, Digits
from sklearn.metrics import f1_score
from datasets import load_dataset
import numpy as np
import torch
import random
import json


In [2]:
# Get the current accelerator.
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else 'cpu'
device

device(type='cuda')

In [3]:
# Load the dataset.
dataset = load_dataset('maliknaik/natural_unit_conversion')
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'entities'],
        num_rows: 583863
    })
    validation: Dataset({
        features: ['text', 'entities'],
        num_rows: 100091
    })
    test: Dataset({
        features: ['text', 'entities'],
        num_rows: 150137
    })
})

In [4]:
# Print first training sample
text = dataset['train'][0]['text']
print(f"Text: {text}")

print("Entities:")
for entity in dataset['train'][0]['entities']:
    word = text[entity['start']:entity['end']]
    print(f"Word: \"{word}\", Tag: {entity['tag']}")

Text: Need the value of 6974 degrees f in degrees de please
Entities:
Word: "degrees f", Tag: FROM_UNIT
Word: "6974", Tag: UNIT_VALUE
Word: "degrees de", Tag: TO_UNIT


In [5]:
# Set new labels for the model.

label2id = {
 'O': 0,
 'B-FROM_UNIT': 1,
 'I-FROM_UNIT': 2,
 'B-TO_UNIT': 3,
 'I-TO_UNIT': 4,
 'B-FEET_VALUE': 5,
 'I-FEET_VALUE': 6,
 'B-INCH_VALUE': 7,
 'I-INCH_VALUE': 8,
}

# Update the id2lable for the model.
id2label = {v: k for k, v in label2id.items()}

In [6]:
model_name = 'distilbert/distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name, split_special_tokens=True)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=len(label2id),
    label2id=label2id, 
    id2label=id2label
).to(device)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
len(tokenizer)

30522

In [8]:
model.resize_token_embeddings(len(tokenizer))

Embedding(30522, 768, padding_idx=0)

In [9]:
model.config.label2id

{'O': 0,
 'B-FROM_UNIT': 1,
 'I-FROM_UNIT': 2,
 'B-TO_UNIT': 3,
 'I-TO_UNIT': 4,
 'B-FEET_VALUE': 5,
 'I-FEET_VALUE': 6,
 'B-INCH_VALUE': 7,
 'I-INCH_VALUE': 8}

In [10]:
def debug_aligned_sample(tokenized_input_ids, labels):
    tokens_str = tokenizer.convert_ids_to_tokens(tokenized_input_ids)
    
    for i in range(len(labels)):
        print(f"{tokens_str[i]} \t=>\t {labels[i]} \t=>\t {id2label[labels[i]] if labels[i] > 0 else labels[i]}")

In [11]:
def align_labels_new(texts, entities, debug=False):
    batch_input_ids = []
    batch_attention_masks = []
    batch_labels = []

    for text, entities in zip(texts, entities):
        tokenized_inputs = tokenizer(text, padding=True, truncation=True, return_offsets_mapping=True)
        word_ids = tokenized_inputs.word_ids()
        offset_mapping = tokenized_inputs['offset_mapping']
        tokens = tokenizer.convert_ids_to_tokens(tokenized_inputs['input_ids'])
        labels = [0 for _ in range(len(word_ids))]

        prev_word_id = None
        for i, word_id in enumerate(word_ids):

            if word_id is None:
                labels[i] = -100;
            for entity in entities:
                start, end, tag = entity['start'], entity['end'], entity['tag']
                
                if tag == 'UNIT_VALUE':
                    continue

                os, oe = offset_mapping[i]
                if os >= start and oe <= end:
                    prefix = 'I-' if prev_word_id == word_id else 'B-'
                    labels[i] = label2id[f'{prefix}{tag}']
            prev_word_id = word_id

        batch_input_ids.append(tokenized_inputs['input_ids'])
        batch_attention_masks.append(tokenized_inputs['attention_mask'])
        batch_labels.append(labels)

    if debug:
        n = random.randint(0, len(batch_input_ids) - 1)
        debug_aligned_sample(batch_input_ids[n], batch_labels[n])
        
    max_length = max(len(ids) for ids in batch_input_ids)
    pad_input_ids = [ids + [tokenizer.pad_token_id] * (max_length - len(ids)) for ids in batch_input_ids]
    pad_attention_masks = [ids + [tokenizer.pad_token_id] * (max_length - len(ids)) for ids in batch_attention_masks]
    pad_labels = [ids + [tokenizer.pad_token_id] * (max_length - len(ids)) for ids in batch_labels]

    input_ids = torch.tensor(pad_input_ids)
    attention_masks = torch.tensor(pad_attention_masks)
    labels = torch.tensor(pad_labels)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_masks,
        'labels': labels
    }
    

In [12]:
train_data = dataset['train']
val_data = dataset['validation']

def debug_aligned_labels_new():
    sample_texts, sample_entities = train_data['text'][:5], train_data['entities'][:5]
    return align_labels_new(sample_texts, sample_entities, debug=True)

debug_aligned_labels_new()

[CLS] 	=>	 -100 	=>	 -100
can 	=>	 0 	=>	 0
you 	=>	 0 	=>	 0
convert 	=>	 0 	=>	 0
77 	=>	 0 	=>	 0
##9 	=>	 0 	=>	 0
. 	=>	 0 	=>	 0
98 	=>	 0 	=>	 0
##8 	=>	 0 	=>	 0
ft 	=>	 1 	=>	 B-FROM_UNIT
^ 	=>	 1 	=>	 B-FROM_UNIT
3 	=>	 1 	=>	 B-FROM_UNIT
to 	=>	 0 	=>	 0
imperial 	=>	 3 	=>	 B-TO_UNIT
##fl 	=>	 4 	=>	 I-TO_UNIT
##uid 	=>	 4 	=>	 I-TO_UNIT
##oun 	=>	 4 	=>	 I-TO_UNIT
##ces 	=>	 4 	=>	 I-TO_UNIT
? 	=>	 0 	=>	 0
[SEP] 	=>	 -100 	=>	 -100


{'input_ids': tensor([[  101,  2342,  1996,  3643,  1997,  6353,  2581,  2549,  5445,  1042,
           1999,  5445,  2139,  3531,   102,     0,     0,     0,     0,     0,
              0],
         [  101,  1045,  2342, 19235,  2683, 11919,  5563,  3421,  1999, 10268,
            102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0],
         [  101,  2064,  2017, 10463,  6255,  2683,  1012,  5818,  2620,  3027,
           1034,  1017,  2000,  4461, 10258, 21272, 23709,  9623,  1029,   102,
              0],
         [  101,  1045,  1005,  1049,  5457,  2055,  2129,  2000, 10463,  5139,
           2581,  1012, 17691,  1050, 30141,  2213,  2000,  9044,  2389,  5563,
            102],
         [  101, 10938,  1022,  9017,  2046, 14255,  2497,   102,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
   

In [13]:
tokenizer.pad_token

'[PAD]'

In [14]:
def tokenize_data(data, max_len=None):
    tokenized_data = []

    if max_len is None:
        max_len = len(data)

    texts = data['text'][:max_len]
    entities = data['entities'][:max_len]
    
    return align_labels_new(texts, entities)

In [15]:
len(dataset['train'])

583863

In [16]:
tokenized_train_data = tokenize_data(dataset['train'], max_len=int(len(dataset['train'])))
tokenized_eval_data = tokenize_data(dataset['validation'], max_len=int(len(dataset['validation'])))

In [17]:
from torch.utils.data import Dataset

class TokenClassificationDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):

        return {            
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['labels'][idx]
        }

    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = TokenClassificationDataset(tokenized_train_data)
eval_dataset = TokenClassificationDataset(tokenized_eval_data)

In [18]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Check if labels are multi-output, then flatten them
    if len(labels.shape) > 1:  # Multi-output case
        labels = labels.flatten()
        preds = preds.flatten()

    return {"f1": f1_score(labels, preds, average="weighted")}


In [20]:
training_args = TrainingArguments(
    output_dir='./unit_converter_models',
    per_device_train_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.01,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    save_strategy='epoch',
    use_cpu=False,   
    load_best_model_at_end=True,
    metric_for_best_model="f1",  # Track best model using F1-score
    greater_is_better=True  # Higher F1 score is better
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.000300,0.000267,0.902592
2,0.000200,0.000097,0.905813
3,0.000200,0.000054,0.905868
4,0.000100,0.000199,0.902414
5,0.000000,0.000034,0.901587
6,0.000000,0.000032,0.899184
7,0.000000,0.000013,0.899096
8,0.000000,0.000012,0.902822
9,0.000000,0.000019,0.902470
10,0.000000,0.000015,0.901913


TrainOutput(global_step=91230, training_loss=0.0003901872402552356, metrics={'train_runtime': 22698.0527, 'train_samples_per_second': 257.23, 'train_steps_per_second': 4.019, 'total_flos': 4.470302697589261e+16, 'train_loss': 0.0003901872402552356, 'epoch': 10.0})

In [22]:
best_model_checkpoint = trainer.state.best_model_checkpoint
print(f"Best model checkpoint: {best_model_checkpoint}")

Best model checkpoint: ./unit_converter_models/checkpoint-27369


In [23]:
ner_pipeline = pipeline(
    'ner',
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy='none'
)

Device set to use cuda:0


In [24]:
text = dataset['validation'][0]['text']

print(text)
ner_pipeline(text)

Please help with converting 7894 cubicinches to imperial gills


[{'entity': 'B-FROM_UNIT',
  'score': np.float32(0.99999785),
  'index': 8,
  'word': 'cubic',
  'start': 33,
  'end': 38},
 {'entity': 'I-FROM_UNIT',
  'score': np.float32(0.9999981),
  'index': 9,
  'word': '##in',
  'start': 38,
  'end': 40},
 {'entity': 'I-FROM_UNIT',
  'score': np.float32(0.9999982),
  'index': 10,
  'word': '##ches',
  'start': 40,
  'end': 44},
 {'entity': 'B-TO_UNIT',
  'score': np.float32(0.99999833),
  'index': 12,
  'word': 'imperial',
  'start': 48,
  'end': 56},
 {'entity': 'B-TO_UNIT',
  'score': np.float32(0.99999833),
  'index': 13,
  'word': 'gills',
  'start': 57,
  'end': 62}]

In [25]:
model.eval()

DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

### Model quantization to compress the model

In [26]:
# Converting float model to a dynamic weight-only quantized model.
quantized_model_qint8 = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

torch.save(quantized_model_qint8.state_dict(), './unit_converter_models/quanited_model_qint8.pt')

In [28]:
import os

print(f'Original Model size: {os.path.getsize(best_model_checkpoint + '/model.safetensors') / (1024 * 1024)} MB')
print(f'Quantized Model size: {os.path.getsize('./unit_converter_models/quanited_model_qint8.pt') / (1024 * 1024)} MB')

Original Model size: 253.1924705505371 MB
Quantized Model size: 131.7301540374756 MB
